# CAM:macro_microphysics on its own

`macro_microphysics` is CAM5's cloud macrophysics + microphysics stage, one
box of the PI-atm workflow.  freeCAM can call it alone: the cell below starts
a model, writes whatever it likes into the live StatePool, runs only that
process, and reports every field the call changed.

Change the two input lines to set different inputs; everything else stays the
same.  The model keeps running afterwards, so the cell can be re-executed
against the state the previous call left behind.


In [ ]:
import freecam as fc
import numpy as np

# CAM:macro_microphysics -- CAM5's cloud macrophysics + microphysics stage,
# called on its own.  Start one model, set whatever inputs you like, run only
# this process, and read back everything it produced.
driver = fc.Driver(case='PI-atm', nsteps=1)
driver.initialize()
driver.run(steps=1)  # one complete CAM step, so the stage starts from a real state

state = driver.cam.state
macro_microphysics = driver.cam.workflow['macro_microphysics']

# ---- inputs: any StatePool field, any value ------------------------------
state.T += 2.0                # 2 K warmer everywhere
state.q[:, :, 0, :] *= 1.05   # 5% more water vapour (constituent 0 is Q)

# ---- run this one process, and nothing else ------------------------------
watched = tuple(
    row['name']
    for row in state.describe()
    if np.dtype(row['dtype']).kind == 'f'
    and row['name'].partition('.')[0] in {'phys_state', 'phys_tend', 'cam_out'}
)
before = {name: state[name].get(rank=0) for name in watched}
temperature_before = state.T.mean(rank='global')

macro_microphysics.run()

after = {name: state[name].get(rank=0) for name in watched}
temperature_after = state.T.mean(rank='global')

# ---- outputs: every field this one call wrote ----------------------------
# CAM pads each chunk to pcols columns and leaves the padding uninitialized,
# so only finite entries are compared.
def largest_change(name):
    delta = np.abs(after[name] - before[name])
    finite = delta[np.isfinite(delta)]
    return float(finite.max()) if finite.size else 0.0


outputs = {
    'global mean T (K)': f'{temperature_before:.4f} -> {temperature_after:.4f}',
}
outputs.update({
    name: {
        'max |change| on rank 0': largest_change(name),
        'global mean after': state[name].mean(rank='global'),
    }
    for name in watched
    if largest_change(name) > 0.0
})
# Constituents 0-4 are Q, CLDLIQ, CLDICE, NUMLIQ, NUMICE; the rest are the
# iCESM water-isotope tracers that follow them.
dq = np.abs(after['phys_state.q'] - before['phys_state.q'])
outputs['phys_state.q']['changed constituents'] = {
    index: float(np.nanmax(dq[:, :, index, :]))
    for index in range(dq.shape[2])
    if np.nanmax(dq[:, :, index, :]) > 0.0
}
display(outputs)

# The model stays live: re-run this cell against the state it just produced,
# or call driver.close() to release the MPI ranks.
